# Klasifikasi Citra X-Ray Penyakit Paru-Paru dengan PyTorch
Oleh Muhammad Fayaz Irham (1240901103003)

Notebook ini membandingkan empat arsitektur Deep Learning berbasis **PyTorch** untuk mengklasifikasikan citra X-Ray dada ke dalam tiga kelas (`Lung_Opacity`, `Normal`, dan `Viral Pneumonia`).
1. **EnhancedResidualSE_CNN** — Arsitektur Custom CNN berbasis Residual Connections, Squeeze-and-Excitation Channel Attention, aktivasi SiLU, Dual Global Pooling, dan Test-Time Augmentation.
2. **EfficientNetV2B0** — Transfer Learning dan Fine-Tuning menggunakan `timm`.
3. **ConvNeXtTiny** — Transfer Learning dan Fine-Tuning menggunakan `timm`.
4. **Xception** — Transfer Learning dan Fine-Tuning menggunakan `timm`.

**Prasyarat Sesi Kaggle:**
* Tambahkan dataset `fatemehmehrparvar/lung-disease` pada panel **Input**.
* Aktifkan **Accelerator: GPU P100 / GPU T4** dan **Internet: On** pada panel **Session options**.
**Konfigurasi:** secara default input dibaca dari `/kaggle/input` dan hasil ditulis ke `/kaggle/working/outputs`. Atur `LUNG_CNN_INPUT_DIR` dan `LUNG_CNN_OUTPUT_DIR` sebelum menjalankan notebook untuk memakai direktori lokal.


## 1. Konfigurasi Kaggle, Seed, dan GPU CUDA

In [ ]:
import os
import gc
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageEnhance

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm

# Konfigurasi global
SEED = 999
IMG_SIZE = 224
NUM_CLASSES = 3
BATCH_SIZE = 32
NUM_WORKERS = 2
CUSTOM_CNN_EPOCHS = 30
TL_WARMUP_EPOCHS = 5
TL_FINETUNE_EPOCHS = 15

def seed_everything(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

# Direktori input dan output Kaggle
INPUT_DIR = Path(os.environ.get('LUNG_CNN_INPUT_DIR', '/kaggle/input'))
OUTPUT_DIR = Path(os.environ.get('LUNG_CNN_OUTPUT_DIR', '/kaggle/working/outputs'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Konfigurasi perangkat komputasi GPU Kaggle
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'

print(f"PyTorch Version  : {torch.__version__}")
print(f"timm Version     : {timm.__version__}")
print(f"Perangkat        : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU Aktif        : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Tersedia    : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("CUDA tidak tersedia; pelatihan CPU dapat berjalan sangat lambat.")
print(f"Batch Size       : {BATCH_SIZE}")
print(f"Direktori Input  : {INPUT_DIR}")
print(f"Direktori Output : {OUTPUT_DIR}")

## 2. Pemuatan Dataset dari `/kaggle/input` & Stratified Split

In [ ]:
TARGET_CLASSES = {'Lung_Opacity', 'Normal', 'Viral Pneumonia'}
VALID_EXTS = {'.png', '.jpg', '.jpeg', '.bmp'}

data_dir = None
for root, dirs, _ in os.walk(INPUT_DIR):
    if TARGET_CLASSES.issubset(set(dirs)):
        data_dir = Path(root)
        break

if data_dir is None:
    raise FileNotFoundError(
        "Dataset tidak ditemukan di /kaggle/input. Pastikan dataset 'fatemehmehrparvar/lung-disease' sudah ditambahkan."
    )

print(f"Direktori dataset aktif: {data_dir}")

class_names = sorted(list(TARGET_CLASSES))
class_to_idx = {name: idx for idx, name in enumerate(class_names)}

records = []
for cls_name in class_names:
    cls_folder = data_dir / cls_name
    for file_path in sorted(cls_folder.iterdir()):
        if file_path.suffix.lower() in VALID_EXTS:
            records.append({
                'filepath': str(file_path),
                'label_name': cls_name,
                'label': class_to_idx[cls_name]
            })

df = pd.DataFrame(records)
print(f"Total citra valid: {len(df)}")
print("\nDistribusi citra per kelas:")
print(df['label_name'].value_counts())

# Pembagian terstratifikasi: 70% Train, 15% Validation, 15% Test
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df['label'], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED
)

print(f"\nTrain set      : {len(train_df)} citra")
print(f"Validation set : {len(val_df)} citra")
print(f"Test set       : {len(test_df)} citra")

## 3. Dataset & Citra Medis
Dataset mendukung parameter `mean`, `std`, dan `img_size` yang dapat disesuaikan otomatis mengikuti konfigurasi setiap arsitektur pretrained (`EfficientNetV2B0`, `ConvNeXtTiny`, maupun `Xception`).

In [ ]:
DEFAULT_MEAN = (0.485, 0.456, 0.406)
DEFAULT_STD = (0.229, 0.224, 0.225)

class LungXRayDataset(Dataset):
    def __init__(self, dataframe, img_size=IMG_SIZE, mean=DEFAULT_MEAN, std=DEFAULT_STD, is_train=False, tta_mode=None):
        self.filepaths = dataframe['filepath'].values
        self.labels = dataframe['label'].values
        self.img_size = img_size
        self.mean = np.array(mean, dtype=np.float32)
        self.std = np.array(std, dtype=np.float32)
        self.is_train = is_train
        self.tta_mode = tta_mode

    def __len__(self):
        return len(self.filepaths)

    def _augment_pil(self, img):
        if random.random() < 0.7:
            angle = random.uniform(-8.0, 8.0)
            img = img.rotate(angle, resample=Image.BILINEAR, fillcolor=(0, 0, 0))

        if random.random() < 0.6:
            w, h = img.size
            scale = random.uniform(0.90, 1.0)
            nw, nh = int(w * scale), int(h * scale)
            left = random.randint(0, w - nw)
            top = random.randint(0, h - nh)
            img = img.crop((left, top, left + nw, top + nh)).resize(
                (self.img_size, self.img_size), Image.BILINEAR
            )

        if random.random() < 0.6:
            img = ImageEnhance.Contrast(img).enhance(random.uniform(0.88, 1.15))
        if random.random() < 0.5:
            img = ImageEnhance.Brightness(img).enhance(random.uniform(0.90, 1.12))

        return img

    def _apply_tta(self, img):
        if self.tta_mode == 'contrast_up':
            return ImageEnhance.Contrast(img).enhance(1.08)
        elif self.tta_mode == 'brightness_up':
            return ImageEnhance.Brightness(img).enhance(1.05)
        elif self.tta_mode == 'zoom_center':
            w, h = img.size
            margin = int(w * 0.04)
            return img.crop((margin, margin, w - margin, h - margin)).resize(
                (self.img_size, self.img_size), Image.BILINEAR
            )
        return img

    def __getitem__(self, idx):
        img = Image.open(self.filepaths[idx]).convert('RGB')
        img = img.resize((self.img_size, self.img_size), Image.BILINEAR)

        if self.is_train:
            img = self._augment_pil(img)
        elif self.tta_mode is not None:
            img = self._apply_tta(img)

        arr = np.asarray(img, dtype=np.float32) / 255.0
        arr = (arr - self.mean) / self.std

        if self.is_train and random.random() < 0.35:
            eh = random.randint(12, 28)
            ew = random.randint(12, 28)
            y0 = random.randint(0, self.img_size - eh)
            x0 = random.randint(0, self.img_size - ew)
            arr[y0:y0 + eh, x0:x0 + ew, :] = 0.0

        tensor_img = torch.from_numpy(np.transpose(arr, (2, 0, 1)))
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return tensor_img, label

def create_dataloaders(img_size=IMG_SIZE, mean=DEFAULT_MEAN, std=DEFAULT_STD):
    train_ds = LungXRayDataset(train_df, img_size=img_size, mean=mean, std=std, is_train=True)
    val_ds = LungXRayDataset(val_df, img_size=img_size, mean=mean, std=std, is_train=False)
    test_ds = LungXRayDataset(test_df, img_size=img_size, mean=mean, std=std, is_train=False)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    return train_loader, val_loader, test_loader

## 4. Pelatihan, Evaluasi, dan Test-Time Augmentation

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

@torch.no_grad()
def evaluate_loader(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total

@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    all_probs = []
    for images, _ in loader:
        images = images.to(DEVICE, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            logits = model(images)
            probs = F.softmax(logits, dim=1)
        all_probs.append(probs.float().cpu().numpy())
    return np.concatenate(all_probs, axis=0)

def predict_with_tta(model, img_size=IMG_SIZE, mean=DEFAULT_MEAN, std=DEFAULT_STD):
    tta_modes = [None, 'contrast_up', 'brightness_up', 'zoom_center']
    probs_list = []
    for mode in tta_modes:
        ds = LungXRayDataset(test_df, img_size=img_size, mean=mean, std=std, is_train=False, tta_mode=mode)
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
        probs_list.append(predict_probs(model, loader))
    return np.mean(probs_list, axis=0)

def cleanup_gpu(*objs):
    for obj in objs:
        del obj
    gc.collect()
    torch.cuda.empty_cache()

## 5. Custom CNN (`EnhancedResidualSE_CNN`)
Arsitektur ini dirancang khusus untuk mengatasi *underfitting* pada model baseline melalui 4 tahap blok konvolusi residual (`64 -> 128 -> 256 -> 512`), kanal atensi *Squeeze-and-Excitation*, aktivasi *SiLU*, serta gabungan *AdaptiveAvgPool2d* dan *AdaptiveMaxPool2d*.

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction, bias=False)
        self.fc2 = nn.Linear(channels // reduction, channels, bias=False)

    def forward(self, x):
        b, c, _, _ = x.size()
        y = x.mean(dim=(2, 3))
        y = F.silu(self.fc1(y))
        y = torch.sigmoid(self.fc2(y)).view(b, c, 1, 1)
        return x * y

class ResidualSEBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1, drop_rate=0.1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.se = SEBlock(out_c)
        self.drop = nn.Dropout2d(drop_rate) if drop_rate > 0 else nn.Identity()

        self.shortcut = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_c)
            )

    def forward(self, x):
        out = F.silu(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        return F.silu(out + self.shortcut(x))

class EnhancedResidualSE_CNN(nn.Module):
    def __init__(self, num_classes=3, dropout_rate=0.35):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.SiLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.SiLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.stage1 = nn.Sequential(
            ResidualSEBlock(64, 64, stride=1, drop_rate=0.05),
            ResidualSEBlock(64, 64, stride=1, drop_rate=0.05)
        )
        self.stage2 = nn.Sequential(
            ResidualSEBlock(64, 128, stride=2, drop_rate=0.10),
            ResidualSEBlock(128, 128, stride=1, drop_rate=0.10)
        )
        self.stage3 = nn.Sequential(
            ResidualSEBlock(128, 256, stride=2, drop_rate=0.15),
            ResidualSEBlock(256, 256, stride=1, drop_rate=0.15)
        )
        self.stage4 = nn.Sequential(
            ResidualSEBlock(256, 512, stride=2, drop_rate=0.15),
            ResidualSEBlock(512, 512, stride=1, drop_rate=0.15)
        )
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(512 * 2),
            nn.Dropout(dropout_rate),
            nn.Linear(512 * 2, 256, bias=False),
            nn.BatchNorm1d(256),
            nn.SiLU(inplace=True),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        feat = torch.cat([self.avg_pool(x), self.max_pool(x)], dim=1)
        return self.classifier(feat)

# Penyimpanan hasil seluruh model
histories = {}
eval_results = []
predictions_dict = {}
y_true = test_df['label'].values

print("="*65)
print("Melatih Model 1: EnhancedResidualSE_CNN")
print("="*65)

train_loader, val_loader, test_loader = create_dataloaders(IMG_SIZE, DEFAULT_MEAN, DEFAULT_STD)
custom_model = EnhancedResidualSE_CNN(num_classes=NUM_CLASSES, dropout_rate=0.35).to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(custom_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CUSTOM_CNN_EPOCHS, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

hist = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
best_ckpt = OUTPUT_DIR / 'EnhancedResidualSE_CNN.pt'

for epoch in range(1, CUSTOM_CNN_EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(custom_model, train_loader, criterion, optimizer, scaler)
    va_loss, va_acc = evaluate_loader(custom_model, val_loader, criterion)
    scheduler.step()

    hist['train_loss'].append(tr_loss)
    hist['train_acc'].append(tr_acc)
    hist['val_loss'].append(va_loss)
    hist['val_acc'].append(va_acc)

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save(custom_model.state_dict(), best_ckpt)

    print(
        f"[EnhancedResidualSE_CNN] Epoch [{epoch:02d}/{CUSTOM_CNN_EPOCHS:02d}] ({time.time()-t0:.1f}s) | "
        f"Train Acc: {tr_acc:.4f} | Val Acc: {va_acc:.4f}"
    )

custom_model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE, weights_only=True))
probs_tta = predict_with_tta(custom_model, IMG_SIZE, DEFAULT_MEAN, DEFAULT_STD)
preds_tta = np.argmax(probs_tta, axis=1)

acc = accuracy_score(y_true, preds_tta)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, preds_tta, average='macro')

histories['EnhancedResidualSE_CNN'] = hist
predictions_dict['EnhancedResidualSE_CNN'] = preds_tta
eval_results.append({
    'Model': 'EnhancedResidualSE_CNN',
    'Test Accuracy': round(acc, 4),
    'Macro Precision': round(prec, 4),
    'Macro Recall': round(rec, 4),
    'Macro F1-Score': round(f1, 4),
    'Total Params': sum(p.numel() for p in custom_model.parameters()),
    'Checkpoint': str(best_ckpt)
})

print(f"\n[EnhancedResidualSE_CNN] Test Accuracy (+TTA): {acc:.4f} ({acc*100:.2f}%)")
print(classification_report(y_true, preds_tta, target_names=class_names, digits=4))

cleanup_gpu(custom_model, optimizer, scheduler, scaler)

## 6. Transfer Learning (`EfficientNetV2B0`, `ConvNeXtTiny`, `Xception`)
Ketiga arsitektur dimuat melalui pustaka `timm` dengan parameter normalisasi (`mean`, `std`) dan ukuran input yang diekstrak secara otomatis dari konfigurasi bawaan tiap model. Pelatihan dijalankan dalam dua tahap:
1. **Tahap 1 — Warmup Classifier Head:** Membekukan seluruh bobot backbone agar lapisan klasifikasi baru beradaptasi dengan stabil.
2. **Tahap 2 — Fine-Tuning Menyeluruh:** Membuka seluruh lapisan jaringan menggunakan *learning rate* kecil (`1e-4`) dengan penjadwal *CosineAnnealingLR*.

In [ ]:
TRANSFER_MODELS = {
    'EfficientNetV2B0': ['tf_efficientnetv2_b0', 'efficientnetv2_b0'],
    'ConvNeXtTiny': ['convnext_tiny'],
    'Xception': ['xception', 'legacy_xception']
}

def load_timm_model(candidates, num_classes=NUM_CLASSES, drop_rate=0.3):
    last_err = None
    for model_id in candidates:
        try:
            model = timm.create_model(
                model_id,
                pretrained=True,
                num_classes=num_classes,
                drop_rate=drop_rate
            )
            return model, model_id
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Gagal memuat model dari kandidat {candidates}: {last_err}")

for display_name, model_candidates in TRANSFER_MODELS.items():
    print("\n" + "="*65)
    print(f"Memulai Transfer Learning: {display_name}")
    print("="*65)

    model, resolved_id = load_timm_model(model_candidates, num_classes=NUM_CLASSES, drop_rate=0.3)
    data_cfg = timm.data.resolve_model_data_config(model)
    model_img_size = data_cfg['input_size'][-1]
    model_mean = data_cfg['mean']
    model_std = data_cfg['std']

    print(f"Model ID timm   : {resolved_id}")
    print(f"Input Resolution: {model_img_size}x{model_img_size}")
    print(f"Normalization   : mean={model_mean}, std={model_std}")

    train_loader, val_loader, test_loader = create_dataloaders(model_img_size, model_mean, model_std)
    model = model.to(DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    hist = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    best_ckpt = OUTPUT_DIR / f"{display_name}.pt"

    # Tahap 1: Warmup Classifier Head (Backbone dibekukan)
    for param in model.parameters():
        param.requires_grad = False
    for param in model.get_classifier().parameters():
        param.requires_grad = True

    optimizer = torch.optim.AdamW(model.get_classifier().parameters(), lr=1e-3, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    for epoch in range(1, TL_WARMUP_EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        va_loss, va_acc = evaluate_loader(model, val_loader, criterion)

        hist['train_loss'].append(tr_loss)
        hist['train_acc'].append(tr_acc)
        hist['val_loss'].append(va_loss)
        hist['val_acc'].append(va_acc)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), best_ckpt)

        print(
            f"[{display_name} | Warmup {epoch:02d}/{TL_WARMUP_EPOCHS:02d}] ({time.time()-t0:.1f}s) | "
            f"Train Acc: {tr_acc:.4f} | Val Acc: {va_acc:.4f}"
        )

    # Tahap 2: Fine-Tuning seluruh lapisan
    for param in model.parameters():
        param.requires_grad = True

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TL_FINETUNE_EPOCHS, eta_min=1e-6)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    for epoch in range(1, TL_FINETUNE_EPOCHS + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        va_loss, va_acc = evaluate_loader(model, val_loader, criterion)
        scheduler.step()

        hist['train_loss'].append(tr_loss)
        hist['train_acc'].append(tr_acc)
        hist['val_loss'].append(va_loss)
        hist['val_acc'].append(va_acc)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            torch.save(model.state_dict(), best_ckpt)

        print(
            f"[{display_name} | FineTune {epoch:02d}/{TL_FINETUNE_EPOCHS:02d}] ({time.time()-t0:.1f}s) | "
            f"Train Acc: {tr_acc:.4f} | Val Acc: {va_acc:.4f}"
        )

    # Evaluasi pada Test Set dengan Test-Time Augmentation
    model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE, weights_only=True))
    probs_tta = predict_with_tta(model, model_img_size, model_mean, model_std)
    preds_tta = np.argmax(probs_tta, axis=1)

    acc = accuracy_score(y_true, preds_tta)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, preds_tta, average='macro')

    histories[display_name] = hist
    predictions_dict[display_name] = preds_tta
    eval_results.append({
        'Model': display_name,
        'Test Accuracy': round(acc, 4),
        'Macro Precision': round(prec, 4),
        'Macro Recall': round(rec, 4),
        'Macro F1-Score': round(f1, 4),
        'Total Params': sum(p.numel() for p in model.parameters()),
        'Checkpoint': str(best_ckpt)
    })

    print(f"\n[{display_name}] Test Accuracy (+TTA): {acc:.4f} ({acc*100:.2f}%)")
    print(classification_report(y_true, preds_tta, target_names=class_names, digits=4))

    cleanup_gpu(model, optimizer, scheduler, scaler)

## 7. Perbandingan Komparatif Keempat Model & Visualisasi Hasil

In [ ]:
summary_df = pd.DataFrame(eval_results).sort_values(by='Macro F1-Score', ascending=False).reset_index(drop=True)
summary_df['Selisih vs Baseline (89.60%)'] = summary_df['Test Accuracy'].apply(lambda x: f"{(x - 0.8960)*100:+.2f}%")

display_cols = ['Model', 'Test Accuracy', 'Selisih vs Baseline (89.60%)', 'Macro Precision', 'Macro Recall', 'Macro F1-Score', 'Total Params']
summary_df[display_cols].to_csv(OUTPUT_DIR / 'pytorch_model_comparison.csv', index=False)

print("=== Ringkasan Perbandingan Arsitektur PyTorch ===")
display(summary_df[display_cols])

In [ ]:
# Plot Kurva Training vs Validation Accuracy untuk keempat arsitektur
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for idx, (name, hist) in enumerate(histories.items()):
    ax = axes[idx]
    epochs_range = range(1, len(hist['train_acc']) + 1)
    ax.plot(epochs_range, hist['train_acc'], label='Train Accuracy', linewidth=2)
    ax.plot(epochs_range, hist['val_acc'], label='Val Accuracy', linewidth=2, linestyle='--')
    ax.set_title(f"{name} - Accuracy Curve")
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend(loc='lower right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pytorch_training_curves.png', dpi=150)
plt.show()

# Plot Confusion Matrix 2x2 untuk keempat arsitektur
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (name, y_pred) in enumerate(predictions_dict.items()):
    cm = confusion_matrix(y_true, y_pred)
    acc_val = accuracy_score(y_true, y_pred) * 100
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=class_names, yticklabels=class_names,
        ax=axes[idx]
    )
    axes[idx].set_title(f"Confusion Matrix - {name} (Acc: {acc_val:.2f}%)")
    axes[idx].set_xlabel('Predicted Label')
    axes[idx].set_ylabel('True Label')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pytorch_confusion_matrices.png', dpi=150)
plt.show()

## 8. Informasi Model Terbaik

In [ ]:
best_row = summary_df.iloc[0]
print(f"Model Terbaik (Macro F1-Score tertinggi) : {best_row['Model']}")
print(f"Test Accuracy (+TTA)                     : {best_row['Test Accuracy']*100:.2f}% ({best_row['Selisih vs Baseline (89.60%)']} vs Baseline)")
print(f"Macro F1-Score                           : {best_row['Macro F1-Score']:.4f}")
print(f"File Bobot Tersimpan                     : {best_row['Checkpoint']}")